In [6]:
import os
import boto3
import duckdb

In [7]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())
print("Connected.")

Connected.


In [8]:
con.execute("""
    CREATE TABLE gold.coco_training AS
    SELECT image_uri, category, bbox_x, bbox_y, bbox_w, bbox_h, split
    FROM silver.coco_annotations
""")
print("gold.coco_training created.")

gold.coco_training created.


In [9]:
con.execute("""
    CREATE TABLE gold.visdrone_training AS
    SELECT clip_uri, fragment_id, frame_id, x, y, w, h, category
    FROM silver.visdrone_annotations
    WHERE category != 'ignored'
""")
print("gold.visdrone_training created.")

gold.visdrone_training created.


In [10]:
result = con.sql("""
    SELECT clip_uri, fragment_id, start_frame, end_frame, n_objects, classes
    FROM silver.visdrone_fragments
    WHERE n_objects > 20
    ORDER BY n_objects DESC
    LIMIT 10
""").df()
print(result)

                                            clip_uri  fragment_id  \
0  s3://lakehouse/assets/visdrone/frames/uav00001...            4   
1  s3://lakehouse/assets/visdrone/frames/uav00001...            3   
2  s3://lakehouse/assets/visdrone/frames/uav00001...            0   
3  s3://lakehouse/assets/visdrone/frames/uav00001...            2   
4  s3://lakehouse/assets/visdrone/frames/uav00001...            1   
5  s3://lakehouse/assets/visdrone/frames/uav00001...            5   
6  s3://lakehouse/assets/visdrone/frames/uav00001...            6   
7  s3://lakehouse/assets/visdrone/frames/uav00001...           10   
8  s3://lakehouse/assets/visdrone/frames/uav00001...            9   
9  s3://lakehouse/assets/visdrone/frames/uav00001...            8   

   start_frame  end_frame  n_objects  \
0          121        150       3366   
1           91        120       3360   
2            1         30       3248   
3           61         90       3177   
4           31         60       3143   


In [11]:
S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://rustfs:9000")
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

for _, row in result.iterrows():
    # clip_uri is s3://lakehouse/assets/visdrone/frames/<seq>/
    prefix = row["clip_uri"].replace("s3://lakehouse/", "")
    seq = prefix.rstrip("/").split("/")[-1]
    for frame_id in range(row["start_frame"], row["end_frame"] + 1):
        key = f"{prefix}{str(frame_id).zfill(7)}.jpg"
        try:
            s3.head_object(Bucket="lakehouse", Key=key)
            print(f"EXISTS: {key}")
        except:
            print(f"MISSING: {key}")
    break  # just verify first fragment to keep output short

EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000121.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000122.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000123.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000124.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000125.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000126.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000127.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000128.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000129.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000130.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000131.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000132.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000133.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000134.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000135.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000136.jpg
EXISTS: 

In [14]:
con.sql("""
    SELECT image_uri, COUNT(*) AS n_people
    FROM silver.coco_annotations
    WHERE category = 0
    GROUP BY image_uri
    HAVING COUNT(*) >= 5
    ORDER BY n_people DESC
    LIMIT 10
""").df()

,image_uri,n_people
0,s3://lakehouse/assets/coco/images/139099.jpg,14
1,s3://lakehouse/assets/coco/images/328430.jpg,14
2,s3://lakehouse/assets/coco/images/361730.jpg,14
3,s3://lakehouse/assets/coco/images/363188.jpg,14
4,s3://lakehouse/assets/coco/images/84031.jpg,14
5,s3://lakehouse/assets/coco/images/338219.jpg,14
6,s3://lakehouse/assets/coco/images/463522.jpg,14
7,s3://lakehouse/assets/coco/images/463542.jpg,14
8,s3://lakehouse/assets/coco/images/541123.jpg,14
9,s3://lakehouse/assets/coco/images/37689.jpg,14


In [15]:
print(con.sql("FROM ducklake_snapshots('lake')").df())

    snapshot_id                    snapshot_time  schema_version  \
0             0 2026-06-25 19:47:58.592626+00:00               0   
1             1 2026-06-25 19:47:58.638004+00:00               1   
2             2 2026-06-25 19:47:58.647187+00:00               2   
3             3 2026-06-25 19:47:58.669349+00:00               3   
4             4 2026-06-25 22:18:44.460503+00:00               4   
5             5 2026-06-25 23:13:50.427815+00:00               5   
6             6 2026-06-25 23:13:50.502705+00:00               6   
7             7 2026-06-26 00:31:41.238716+00:00               7   
8             8 2026-06-26 00:31:56.698735+00:00               8   
9             9 2026-06-26 00:32:06.396628+00:00               9   
10           10 2026-06-26 01:47:33.156378+00:00              10   
11           11 2026-06-26 01:47:40.993846+00:00              11   

                                              changes author commit_message  \
0                       {'schemas_cr

In [21]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())
con.execute("""
    CREATE TABLE silver.coco_annotations AS
    SELECT DISTINCT ON (bbox_id)
        image_uri,
        image_id,
        width,
        height,
        bbox_id,
        CASE category
            WHEN 0 THEN 'person' WHEN 1 THEN 'bicycle' WHEN 2 THEN 'car'
            WHEN 3 THEN 'motorcycle' WHEN 4 THEN 'airplane' WHEN 5 THEN 'bus'
            WHEN 6 THEN 'train' WHEN 7 THEN 'truck' WHEN 8 THEN 'boat'
            WHEN 9 THEN 'traffic light' WHEN 10 THEN 'fire hydrant'
            WHEN 11 THEN 'stop sign' WHEN 12 THEN 'parking meter'
            WHEN 13 THEN 'bench' WHEN 14 THEN 'bird' WHEN 15 THEN 'cat'
            WHEN 16 THEN 'dog' WHEN 17 THEN 'horse' WHEN 18 THEN 'sheep'
            WHEN 19 THEN 'cow' WHEN 20 THEN 'elephant' WHEN 21 THEN 'bear'
            WHEN 22 THEN 'zebra' WHEN 23 THEN 'giraffe' WHEN 24 THEN 'backpack'
            WHEN 25 THEN 'umbrella' WHEN 26 THEN 'handbag' WHEN 27 THEN 'tie'
            WHEN 28 THEN 'suitcase' WHEN 29 THEN 'frisbee' WHEN 30 THEN 'skis'
            WHEN 31 THEN 'snowboard' WHEN 32 THEN 'sports ball' WHEN 33 THEN 'kite'
            WHEN 34 THEN 'baseball bat' WHEN 35 THEN 'baseball glove'
            WHEN 36 THEN 'skateboard' WHEN 37 THEN 'surfboard'
            WHEN 38 THEN 'tennis racket' WHEN 39 THEN 'bottle'
            WHEN 40 THEN 'wine glass' WHEN 41 THEN 'cup' WHEN 42 THEN 'fork'
            WHEN 43 THEN 'knife' WHEN 44 THEN 'spoon' WHEN 45 THEN 'bowl'
            WHEN 46 THEN 'banana' WHEN 47 THEN 'apple' WHEN 48 THEN 'sandwich'
            WHEN 49 THEN 'orange' WHEN 50 THEN 'broccoli' WHEN 51 THEN 'carrot'
            WHEN 52 THEN 'hot dog' WHEN 53 THEN 'pizza' WHEN 54 THEN 'donut'
            WHEN 55 THEN 'cake' WHEN 56 THEN 'chair' WHEN 57 THEN 'couch'
            WHEN 58 THEN 'potted plant' WHEN 59 THEN 'bed'
            WHEN 60 THEN 'dining table' WHEN 61 THEN 'toilet' WHEN 62 THEN 'tv'
            WHEN 63 THEN 'laptop' WHEN 64 THEN 'mouse' WHEN 65 THEN 'remote'
            WHEN 66 THEN 'keyboard' WHEN 67 THEN 'cell phone'
            WHEN 68 THEN 'microwave' WHEN 69 THEN 'oven' WHEN 70 THEN 'toaster'
            WHEN 71 THEN 'sink' WHEN 72 THEN 'refrigerator' WHEN 73 THEN 'book'
            WHEN 74 THEN 'clock' WHEN 75 THEN 'vase' WHEN 76 THEN 'scissors'
            WHEN 77 THEN 'teddy bear' WHEN 78 THEN 'hair drier'
            WHEN 79 THEN 'toothbrush'
        END AS category,
        CAST(regexp_extract(bbox, '\\[([0-9.]+),', 1) AS DOUBLE) AS bbox_x,
        CAST(regexp_extract(bbox, '\\[[0-9.]+,\\s*([0-9.]+),', 1) AS DOUBLE) AS bbox_y,
        CAST(regexp_extract(bbox, '\\[[0-9.]+,\\s*[0-9.]+,\\s*([0-9.]+),', 1) AS DOUBLE) AS bbox_w,
        CAST(regexp_extract(bbox, '\\[[0-9.]+,\\s*[0-9.]+,\\s*[0-9.]+,\\s*([0-9.]+)\\]', 1) AS DOUBLE) AS bbox_h,
        area,
        'val' AS split
    FROM raw.coco_annotations
    WHERE area > 0
""")
print("silver.coco_annotations recreated with string categories.")

silver.coco_annotations recreated with string categories.


In [22]:
con.execute("""
    CREATE TABLE gold.coco_training AS
    SELECT image_uri, category, bbox_x, bbox_y, bbox_w, bbox_h, split
    FROM silver.coco_annotations
""")
print("gold.coco_training recreated.")
con.close()

gold.coco_training recreated.
